In [360]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.model_selection import train_test_split, StratifiedKFold

from sklearn.preprocessing import StandardScaler, MinMaxScaler

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import GridSearchCV

from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

from sklearn.base import clone

import warnings
warnings.simplefilter("ignore")

In [315]:
X = pd.read_csv("data/train.csv")
y = X["Survived"]
X_test = pd.read_csv("data/test.csv")
y_test = pd.read_csv("data/gender_submission.csv")["Survived"]
ids = X_test["PassengerId"]

In [316]:
X.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [318]:
# gender OneHotEncoding
ohe = OneHotEncoder(sparse_output=False, drop='first')
encoded = ohe.fit_transform(X[['Sex']])
encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out())
X = X.drop(columns=['Sex'])
X = pd.concat([X, encoded_df], axis=1)

encoded = ohe.transform(X_test[['Sex']])
encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out())
X_test = X_test.drop(columns=['Sex'])
X_test = pd.concat([X_test, encoded_df], axis=1)

In [319]:
# feature engineering
X["family"] = X["SibSp"] + X["Parch"]
X = X.drop(["SibSp", "Parch"], axis=1)

X_test["family"] = X_test["SibSp"] + X_test["Parch"]
X_test = X_test.drop(["SibSp", "Parch"], axis=1)

In [320]:
# fill Age NaN values
imputer = SimpleImputer(strategy='median')
X[['Age', 'Fare']] = imputer.fit_transform(X[['Age', 'Fare']])

X_test[['Age', 'Fare']] = imputer.transform(X_test[['Age', 'Fare']])

In [321]:
# Cabin
X["haveCabin"] = np.where(X["Cabin"].isnull(), 0, 1)
X = X.drop(["Cabin"], axis=1)

X_test["haveCabin"] = np.where(X_test["Cabin"].isnull(), 0, 1)
X_test = X_test.drop(["Cabin"], axis=1)

In [322]:
# drop
del_features = ["PassengerId", "Name", "Ticket", "Embarked"]
X = X.drop(del_features, axis=1)
X = X.drop(["Survived"], axis=1)
X_test = X_test.drop(del_features, axis=1)

In [323]:
# corr_mx = X.corr()
# corr_mx = X.corr(numeric_only=True)
# sns.heatmap(corr_mx, annot=True, fmt=".2f", cmap="coolwarm")

In [324]:
scaler = MinMaxScaler()
X[["Age", "Fare"]] = scaler.fit_transform(X[["Age", "Fare"]])
X_test[["Age", "Fare"]] = scaler.transform(X_test[["Age", "Fare"]])

In [325]:
X.head()

,Pclass,Age,Fare,Sex_male,family,haveCabin
0,3,0.271174,0.014151,1.0,1,0
1,1,0.472229,0.139136,0.0,1,1
2,3,0.321438,0.015469,0.0,0,0
3,1,0.434531,0.103644,0.0,1,1
4,3,0.434531,0.015713,1.0,0,0


## modeling

In [356]:
class Model:
    def __init__(self, model, name=None):
        self.model = model
        self.X_train, self.X_test = X, X_test
        self.y_train, self.y_test = y, y_test
        self.name = name

    def pipeline(self):
        print(f"----- {self.name if self.name else self.model.__class__.__name__} -----")
        pass

    def train(self, X_data=None, y_data=None):
        if X_data is None:
            X_data = self.X_train
        if y_data is None:
            y_data = self.y_train
        
        print("Training...")
        self.model.fit(X_data, y_data)
        print("Train complete")

    def predict(self, data=None):
        if data is None:
            data = self.X_test
        return self.model.predict(data)

    def accuracy(self, pred=None, labels=None):
        if pred is None:
            pred = self.predict()
        if labels is None:
            labels = self.y_test
    
        acc = accuracy_score(labels, pred)
        print(f"acc: {acc:.5f}")
    
    def kfold(self):
        skf = StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=42
        )

        scores = []

        best_score = -1
        best_model = None
        
        for fold, (train_idx, val_idx) in enumerate(skf.split(self.X_train, self.y_train), start=1):

            print(f"-- fold {fold}/5 --")
            
            X_train = self.X_train.iloc[train_idx]
            X_val = self.X_train.iloc[val_idx]
        
            y_train = self.y_train.iloc[train_idx]
            y_val = self.y_train.iloc[val_idx]

            model = clone(self.model)
            model.fit(X_train, y_train)

            pred = model.predict(X_val)
        
            score = accuracy_score(y_val, pred)
            scores.append(score)

            if score > best_score:
                best_score = score
                best_model = model
        
            print(f"accuracy: {score:.4f}\n")

        self.model = best_model
        
        print()
        print(f"Mean accuracy: {np.mean(scores):.4f}")
        print(f"Std: {np.std(scores):.4f}")

#### Random Forest

In [357]:
rf = Model(RandomForestClassifier())

In [358]:
rf.kfold()

-- fold 1/5 --
accuracy: 0.8436

-- fold 2/5 --
accuracy: 0.8483

-- fold 3/5 --
accuracy: 0.7865

-- fold 4/5 --
accuracy: 0.8034

-- fold 5/5 --
accuracy: 0.8315


Mean accuracy: 0.8226
Std: 0.0239


In [359]:
rf.accuracy()

acc: 0.82775


#### Logistic Regression

In [361]:
logreg = Model(LogisticRegression())

In [369]:
logreg.kfold()

-- fold 1/5 --
accuracy: 0.8156

-- fold 2/5 --
accuracy: 0.8034

-- fold 3/5 --
accuracy: 0.7753

-- fold 4/5 --
accuracy: 0.7753

-- fold 5/5 --
accuracy: 0.8427


Mean accuracy: 0.8025
Std: 0.0256


In [370]:
logreg.accuracy()

acc: 0.94019


In [364]:
logreg_pred = logreg.predict()

#### RandomForedt with GridSearchCV

In [374]:
rf = RandomForestClassifier()

rf_grid = {
     'max_depth': [2, 12, 24, None],
     'n_estimators': [5, 25, 50, 100],
}

rf_cv = GridSearchCV(rf, rf_grid, cv=5)

rf = Model(rf_cv)
rf.kfold()
rf.accuracy()

-- fold 1/5 --
accuracy: 0.8436

-- fold 2/5 --
accuracy: 0.8371

-- fold 3/5 --
accuracy: 0.7865

-- fold 4/5 --
accuracy: 0.8090

-- fold 5/5 --
accuracy: 0.8258


Mean accuracy: 0.8204
Std: 0.0206
acc: 0.84689


#### XGBoost

In [375]:
xgb = Model(XGBClassifier())

In [376]:
xgb.kfold()

-- fold 1/5 --
accuracy: 0.8603

-- fold 2/5 --
accuracy: 0.8596

-- fold 3/5 --
accuracy: 0.7753

-- fold 4/5 --
accuracy: 0.8034

-- fold 5/5 --
accuracy: 0.8427


Mean accuracy: 0.8282
Std: 0.0336


In [377]:
xgb.accuracy()

acc: 0.83014


#### XGBppst with GridSearchCV

In [378]:
xgb = XGBClassifier()

xgb_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 6],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}

xgb_cv = GridSearchCV(xgb, xgb_grid, cv=5, n_jobs=-1)

xgb = Model(xgb_cv)
xgb.kfold()
print(f"test accuracy: {xgb.accuracy()}")

-- fold 1/5 --
accuracy: 0.8547

-- fold 2/5 --
accuracy: 0.8820

-- fold 3/5 --
accuracy: 0.8034

-- fold 4/5 --
accuracy: 0.8034

-- fold 5/5 --
accuracy: 0.8596


Mean accuracy: 0.8406
Std: 0.0318
acc: 0.88278
test accuracy: None


In [380]:
xgb.accuracy()

acc: 0.88278


In [381]:
xgb_pred = xgb.predict()

In [383]:
submission = pd.DataFrame({
    'PassengerId': ids,
    'Survived': xgb_pred
})

submission.to_csv('xgb_cv_submission(1).csv', index=False)